In [16]:
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
from torch.utils.data import Dataset
MODEL_NAME = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
df = pd.read_csv("data/voicepay_bio_dataset.csv")
df.head()

,text,amount,currency,recipient,bill_type,intent,tokens,ner_tags
0,سدد فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['سدد', 'فاتورة', 'الكهرباء']","['O', 'O', 'B-BILL_TYPE']"
1,قم بدفع فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['قم', 'بدفع', 'فاتورة', 'الكهرباء']","['O', 'O', 'O', 'B-BILL_TYPE']"
2,ادفع قيمة فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['ادفع', 'قيمة', 'فاتورة', 'الكهرباء']","['O', 'O', 'O', 'B-BILL_TYPE']"
3,قم بسداد فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['قم', 'بسداد', 'فاتورة', 'الكهرباء']","['O', 'O', 'O', 'B-BILL_TYPE']"
4,سدد مستحقات الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['سدد', 'مستحقات', 'الكهرباء']","['O', 'O', 'B-BILL_TYPE']"


In [17]:
import torch
import transformers
import accelerate
import sys

print(sys.executable)
print(torch.__version__)
print(transformers.__version__)
print(accelerate.__version__)

/home/almadhoun/miniconda3/envs/vp_project/bin/python
2.10.0+cu128
5.2.0
1.12.0


In [18]:
# ---- Encode intent labels ----
le = LabelEncoder()
df["label"] = le.fit_transform(df["intent"])
print("Intent classes:", le.classes_)

Intent classes: ['p2p_transfer' 'pay_bill']


In [19]:
# ---- AraBERT tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [20]:
# ---- Dataset class for HuggingFace ----
class NERDataset(Dataset):
    def __init__(self, texts, labels, ner_tags=None, tokenizer=None, max_len=32):
        self.texts = texts
        self.labels = labels
        self.ner_tags = ner_tags
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text.split(),  # tokenize by words
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

        if self.ner_tags is not None:
            # convert NER tags to tensor
            ner_ids = [0 if tag == 'O' else 1 for tag in self.ner_tags[idx]]  # 0=O, 1=BILL_TYPE
            ner_ids = ner_ids + [0]*(self.max_len - len(ner_ids))  # padding
            item["ner_labels"] = torch.tensor(ner_ids, dtype=torch.long)

        return item

# ---- Split train/test ----
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].tolist(), df["label"].tolist(), test_size=0.2, random_state=42
)
ner_train = df["ner_tags"].apply(eval).tolist()
ner_test = df["ner_tags"].apply(eval).tolist()

train_dataset = NERDataset(X_train, y_train, ner_tags=ner_train, tokenizer=tokenizer)
test_dataset = NERDataset(X_test, y_test, ner_tags=ner_test, tokenizer=tokenizer)

In [21]:
from transformers import AutoModelForSequenceClassification

intent_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le.classes_)
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1848.62it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

In [22]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir="./logs",
    logging_steps=10,
    learning_rate=5e-5
)

trainer = Trainer(
    model=intent_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [23]:
# ---- Start training ----
trainer.train()

Step,Training Loss
10,0.710008
20,0.421761
30,0.468787
40,0.080503
50,0.006468
60,0.002250
70,0.001364
80,0.000992
90,0.000668
100,0.000611


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


TrainOutput(global_step=462, training_loss=0.03685952924932524, metrics={'train_runtime': 28.6224, 'train_samples_per_second': 32.282, 'train_steps_per_second': 16.141, 'total_flos': 15194663447040.0, 'train_loss': 0.03685952924932524, 'epoch': 3.0})

In [24]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [25]:
trainer.evaluate()

{'eval_loss': 0.00011114606604678556,
 'eval_runtime': 0.2286,
 'eval_samples_per_second': 336.85,
 'eval_steps_per_second': 170.612,
 'epoch': 3.0}

In [26]:
print(len(train_dataset))
print(len(test_dataset))

308
77


In [27]:
print(df["text"].nunique())

382
